# 03_limpieza_y_preprocesamiento

## Objetivo
Consolidar, limpiar y estandarizar las replies de la recolección formal `media_anchored` sin realizar llamadas a la API de X.

La unidad de análisis es la respuesta/comentario anclado a un post madre de un medio costarricense. El notebook conserva un corpus maestro y genera una vista separada para análisis textual.

## Entradas
- `data/interim/formal_collection/batch_*/replies_clean.csv`
- `data/interim/formal_collection/batch_006/replies_pending.csv`, si existe

## Salidas principales
- `data/interim/formal_collection/replies_formal_combined.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_clean.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_analysis.csv`
- `data/processed/replies_formal_clean.csv`
- `data/processed/replies_formal_analysis.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_training_dedup.csv`
- diagnósticos en `outputs/tables/formal_cleaning_*.csv`

Este flujo no sobrescribe el corpus piloto anterior ni los archivos de etiquetado manual.

## 1. Parámetros metodológicos

In [ ]:
import os

ANALYSIS_LANGS = [
    value.strip().lower()
    for value in os.getenv(
        "ANALYSIS_LANGS", "es,qme,und,qam,qht,zxx"
    ).split(",")
    if value.strip()
]
MIN_TEXT_CHARS = int(os.getenv("MIN_TEXT_CHARS", "3"))
REQUIRE_SUBSTANTIVE_TEXT = os.getenv(
    "REQUIRE_SUBSTANTIVE_TEXT", "true"
).strip().lower() in {"1", "true", "yes", "si"}

print("ANALYSIS_LANGS:", ANALYSIS_LANGS)
print("MIN_TEXT_CHARS:", MIN_TEXT_CHARS)
print("REQUIRE_SUBSTANTIVE_TEXT:", REQUIRE_SUBSTANTIVE_TEXT)
print("API de X: desactivada; este notebook solo lee archivos locales")

## 2. Setup e imports

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if (candidate / "config").exists() and (candidate / "src").exists():
            return candidate
        child = candidate / "HateCR"
        if (child / "config").exists() and (child / "src").exists():
            return child
    raise FileNotFoundError("No se encontró la raíz del proyecto HateCR")


def atomic_to_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    df.to_csv(temporary, index=False)
    temporary.replace(path)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocessing as prep
importlib.reload(prep)

FORMAL_COLLECTION_DIR = PROJECT_ROOT / "data" / "interim" / "formal_collection"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

COMBINED_INPUT_PATH = FORMAL_COLLECTION_DIR / "replies_formal_combined.csv"
MASTER_CLEAN_PATH = PROCESSED_DIR / "x_media_anchored_interactions_corpus_formal_clean.csv"
ANALYSIS_CORPUS_PATH = PROCESSED_DIR / "x_media_anchored_interactions_corpus_formal_analysis.csv"
REPLIES_CLEAN_PATH = PROCESSED_DIR / "replies_formal_clean.csv"
REPLIES_ANALYSIS_PATH = PROCESSED_DIR / "replies_formal_analysis.csv"
TRAINING_DEDUP_PATH = PROCESSED_DIR / "x_media_anchored_interactions_corpus_formal_training_dedup.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("preprocessing module:", prep.__file__)

## 3. Carga y consolidación de batches formales

In [ ]:
raw_df, input_manifest_df = prep.load_formal_reply_batches(FORMAL_COLLECTION_DIR)

if raw_df.empty:
    raise ValueError("No se encontraron batches formales con replies_clean.csv")

print("Batches encontrados:", len(input_manifest_df))
print("Filas consolidadas:", len(raw_df))
print("Replies únicas:", raw_df["reply_id"].nunique(dropna=True))
display(input_manifest_df)

## 4. Validaciones previas

In [ ]:
required_columns = prep.FORMAL_REPLY_REQUIRED_COLUMNS
missing_columns = [column for column in required_columns if column not in raw_df.columns]

assert not missing_columns, f"Faltan columnas: {missing_columns}"
assert raw_df["reply_id"].notna().all(), "Hay reply_id nulos"
assert raw_df["reply_id"].is_unique, "Hay reply_id duplicados entre batches"
assert raw_df["source_type"].eq("reply_to_media_post").all()
assert raw_df["source_universe"].eq("media_anchored").all()
assert raw_df["anchor_media_handle"].notna().all()
assert raw_df["anchor_post_id"].notna().all()
assert raw_df["reply_author_id_hash"].str.fullmatch(r"[0-9a-f]{64}").all()
assert not any(column in raw_df.columns for column in ["username", "user_name", "screen_name"])
assert raw_df["query"].str.match(r"^conversation_id:\d+ -is:retweet$").all()

print("[OK] IDs únicos, anclaje a medios y hashes validados")
print("Eventos:", raw_df["event_id"].nunique())
print("Medios ancla:", raw_df["anchor_media_handle"].nunique())
print("Autores anonimizados:", raw_df["reply_author_id_hash"].nunique())

display(
    raw_df.groupby(["collection_batch_id", "event_id"], dropna=False)
    .size().reset_index(name="n_replies")
)

## 5. Limpieza y normalización

El corpus maestro conserva todos los idiomas reportados por X y añade banderas de calidad. La vista analítica selecciona códigos compatibles con español o idioma incierto y exige contenido textual sustantivo. Esto evita borrar silenciosamente respuestas cortas o mal clasificadas.

In [ ]:
clean_master_df, diagnostics = prep.clean_interactions_corpus(
    corpus_df=raw_df,
    keep_langs=ANALYSIS_LANGS,
    hash_salt=None,
    min_text_chars=MIN_TEXT_CHARS,
    filter_by_lang=False,
)

# El corpus formal no conserva identificadores directos de autor.
clean_master_df = clean_master_df.drop(columns=["author_id"], errors="ignore")
clean_master_df["processing_scope"] = "formal_media_anchored"
clean_master_df["eligible_for_text_analysis"] = (
    clean_master_df["is_lang_kept"]
    & (
        clean_master_df["has_substantive_text"]
        if REQUIRE_SUBSTANTIVE_TEXT
        else True
    )
)

analysis_df = clean_master_df[
    clean_master_df["eligible_for_text_analysis"]
].copy().reset_index(drop=True)

analysis_df["text_duplicate_group_size"] = (
    analysis_df.groupby("text_norm_hash")["tweet_id"].transform("size")
)
analysis_df["text_duplicate_author_count"] = (
    analysis_df.groupby("text_norm_hash")["reply_author_id_hash"].transform("nunique")
)
training_dedup_df = (
    analysis_df.sort_values(["text_norm_hash", "created_at_dt", "tweet_id"])
    .drop_duplicates(subset=["text_norm_hash"], keep="first")
    .reset_index(drop=True)
)

replies_clean_df = prep.build_replies_analysis_view(clean_master_df)
replies_analysis_df = prep.build_replies_analysis_view(analysis_df)

print("Corpus maestro limpio:", len(clean_master_df))
print("Corpus apto para análisis textual:", len(analysis_df))
print("Excluidos solo de la vista analítica:", len(clean_master_df) - len(analysis_df))

## 6. Diagnósticos de calidad

In [ ]:
def compare_group_counts(master, analysis, group_columns):
    master_counts = (
        master.groupby(group_columns, dropna=False)
        .size().reset_index(name="n_master")
    )
    analysis_counts = (
        analysis.groupby(group_columns, dropna=False)
        .size().reset_index(name="n_analysis")
    )
    result = master_counts.merge(analysis_counts, on=group_columns, how="outer")
    result[["n_master", "n_analysis"]] = result[["n_master", "n_analysis"]].fillna(0).astype(int)
    result["analysis_pct"] = (
        result["n_analysis"] / result["n_master"].replace(0, pd.NA) * 100
    ).round(2)
    return result.sort_values("n_master", ascending=False).reset_index(drop=True)


by_batch_df = compare_group_counts(clean_master_df, analysis_df, ["collection_batch_id"])
by_batch_df = by_batch_df.merge(
    input_manifest_df[["collection_batch_id", "batch_status", "pending_source_posts"]],
    on="collection_batch_id",
    how="left",
)
by_event_df = compare_group_counts(clean_master_df, analysis_df, ["event_id", "event_name"])
by_media_df = compare_group_counts(
    clean_master_df, analysis_df, ["anchor_media_id", "anchor_media_handle"]
)
by_event_media_df = compare_group_counts(
    clean_master_df,
    analysis_df,
    ["event_id", "anchor_media_id", "anchor_media_handle"],
)
by_lang_df = compare_group_counts(clean_master_df, analysis_df, ["lang", "lang_group"])

# Las repeticiones textuales se diagnostican, pero no se eliminan del corpus descriptivo.
duplicate_mask = clean_master_df.duplicated("text_norm_hash", keep=False)
duplicate_text_groups_df = (
    clean_master_df.loc[duplicate_mask]
    .groupby("text_norm_hash", dropna=False)
    .agg(
        n_rows=("tweet_id", "size"),
        n_unique_tweets=("tweet_id", "nunique"),
        n_unique_authors=("reply_author_id_hash", "nunique"),
        n_events=("event_id", "nunique"),
        n_anchor_media=("anchor_media_handle", "nunique"),
    )
    .reset_index()
    .sort_values(["n_rows", "n_unique_authors"], ascending=False)
)

exclusions_df = (
    clean_master_df.loc[~clean_master_df["eligible_for_text_analysis"]]
    .assign(
        exclusion_reason=lambda frame: frame.apply(
            lambda row: (
                "other_language_label"
                if not bool(row["is_lang_kept"])
                else "low_information_text"
            ),
            axis=1,
        )
    )
    .groupby(["exclusion_reason", "lang", "lang_group"], dropna=False)
    .size().reset_index(name="n_rows")
    .sort_values("n_rows", ascending=False)
)

missing_core_df = diagnostics["missing_core"].copy()

summary_rows = diagnostics["summary"].to_dict("records")
summary_rows.extend([
    {"metric": "formal_batches_loaded", "value": len(input_manifest_df)},
    {"metric": "formal_batches_partial", "value": int(input_manifest_df["batch_status"].eq("partial").sum())},
    {"metric": "pending_source_posts", "value": int(input_manifest_df["pending_source_posts"].sum())},
    {"metric": "master_clean_rows", "value": len(clean_master_df)},
    {"metric": "analysis_ready_rows", "value": len(analysis_df)},
    {"metric": "analysis_excluded_rows", "value": len(clean_master_df) - len(analysis_df)},
    {"metric": "training_dedup_rows", "value": len(training_dedup_df)},
    {"metric": "training_text_duplicates_removed", "value": len(analysis_df) - len(training_dedup_df)},
    {"metric": "unique_reply_ids", "value": clean_master_df["tweet_id"].nunique()},
    {"metric": "unique_author_hashes", "value": clean_master_df["reply_author_id_hash"].nunique()},
    {"metric": "formal_events", "value": clean_master_df["event_id"].nunique()},
    {"metric": "anchor_media", "value": clean_master_df["anchor_media_handle"].nunique()},
    {"metric": "low_information_rows", "value": int((~clean_master_df["has_substantive_text"]).sum())},
    {"metric": "other_language_label_rows", "value": int((~clean_master_df["is_lang_kept"]).sum())},
    {"metric": "duplicate_text_rows_preserved", "value": int(duplicate_mask.sum())},
    {"metric": "duplicate_text_groups", "value": len(duplicate_text_groups_df)},
])
summary_df = pd.DataFrame(summary_rows)

print("### Resumen")
display(summary_df)
print("### Por evento")
display(by_event_df)
print("### Por idioma")
display(by_lang_df)
print("### Exclusiones de la vista analítica")
display(exclusions_df)

## 7. Exportación reproducible

In [ ]:
output_tables = {
    "formal_cleaning_input_manifest.csv": input_manifest_df,
    "formal_cleaning_summary.csv": summary_df,
    "formal_cleaning_by_batch.csv": by_batch_df,
    "formal_cleaning_by_event.csv": by_event_df,
    "formal_cleaning_by_media.csv": by_media_df,
    "formal_cleaning_by_event_media.csv": by_event_media_df,
    "formal_cleaning_by_lang.csv": by_lang_df,
    "formal_cleaning_missing_core.csv": missing_core_df,
    "formal_cleaning_duplicate_text_groups.csv": duplicate_text_groups_df,
    "formal_cleaning_exclusions.csv": exclusions_df,
}

atomic_to_csv(raw_df, COMBINED_INPUT_PATH)
atomic_to_csv(clean_master_df, MASTER_CLEAN_PATH)
atomic_to_csv(analysis_df, ANALYSIS_CORPUS_PATH)
atomic_to_csv(replies_clean_df, REPLIES_CLEAN_PATH)
atomic_to_csv(replies_analysis_df, REPLIES_ANALYSIS_PATH)
atomic_to_csv(training_dedup_df, TRAINING_DEDUP_PATH)

for filename, table in output_tables.items():
    atomic_to_csv(table, OUTPUT_TABLES_DIR / filename)

print("[OK]", COMBINED_INPUT_PATH)
print("[OK]", MASTER_CLEAN_PATH)
print("[OK]", ANALYSIS_CORPUS_PATH)
print("[OK]", REPLIES_CLEAN_PATH)
print("[OK]", REPLIES_ANALYSIS_PATH)
print("[OK]", TRAINING_DEDUP_PATH)
for filename in output_tables:
    print("[OK]", OUTPUT_TABLES_DIR / filename)

## 8. Validaciones posteriores a la exportación

In [ ]:
master_check_df = pd.read_csv(MASTER_CLEAN_PATH, dtype={"tweet_id": "string"})
analysis_check_df = pd.read_csv(ANALYSIS_CORPUS_PATH, dtype={"tweet_id": "string"})
training_check_df = pd.read_csv(TRAINING_DEDUP_PATH, dtype={"tweet_id": "string"})

assert len(master_check_df) == len(clean_master_df)
assert master_check_df["tweet_id"].is_unique
assert len(analysis_check_df) == len(analysis_df)
assert analysis_check_df["eligible_for_text_analysis"].all()
assert training_check_df["text_norm_hash"].is_unique
assert len(training_check_df) <= len(analysis_check_df)
assert master_check_df["anchor_media_handle"].notna().all()
assert master_check_df["reply_author_id_hash"].str.fullmatch(r"[0-9a-f]{64}").all()
assert not any(column in master_check_df.columns for column in ["author_id", "username", "screen_name"])
assert master_check_df["source_type"].eq("reply_to_media_post").all()
assert master_check_df["event_id"].nunique() == 6

print("[OK] Exportaciones verificadas")
print("Corpus maestro:", len(master_check_df))
print("Corpus analítico:", len(analysis_check_df))
print("Corpus de entrenamiento deduplicado:", len(training_check_df))
print("Replies duplicadas:", int(master_check_df["tweet_id"].duplicated().sum()))
print("Eventos:", master_check_df["event_id"].nunique())
print("Medios ancla:", master_check_df["anchor_media_handle"].nunique())

## 9. Interpretación metodológica

- El corpus maestro conserva todas las replies recuperadas, independientemente de la etiqueta automática de idioma.
- La vista analítica excluye códigos de otros idiomas y textos compuestos únicamente por menciones, enlaces o símbolos.
- `qme`, `und`, `qam`, `qht` y `zxx` se mantienen como categorías inciertas porque X puede asignarlas a respuestas breves en español.
- Las repeticiones de texto se documentan, pero no se eliminan del corpus descriptivo: pueden representar participación coordinada, fórmulas discursivas o respuestas legítimamente repetidas.
- La vista `formal_training_dedup` conserva una sola observación por `text_norm_hash` para reducir fuga de información y sobreponderación durante el entrenamiento supervisado.
- `batch_006` es parcial debido al agotamiento de créditos. El archivo de pendientes debe conservarse para completar la muestra posteriormente.
- La normalización no constituye clasificación de hostilidad ni discurso de odio. La interpretación exige validación manual y contextual.